# NeuroVision-X — Kaggle training driver

Thin driver. No training logic lives here: this notebook attaches the code and the data,
composes the real Hydra config, and calls `scripts/train.py::run_training`. Everything else
is in `src/neurovision/`, tested on the Mac's CPU.

**Before running:** notebook settings → GPU accelerator ON, internet ON (both need a
phone-verified account). Attach the preprocessed dataset. For a long run use
*Save Version → Save & Run All (Commit)*, never the interactive session.

Every cell below cell 1 fails immediately and with a readable message if a path is wrong —
the point is to find out in the first minute, not 40 minutes in.

Full workflow, including how to upload the dataset and chain sessions: `docs/kaggle_workflow.md`.

## 1. Session config — the only cell you edit

In [ ]:
# Set BEFORE torch is imported anywhere (this is the first cell that runs).
# Cheap insurance against allocator fragmentation: probe v2 lost 888 MiB to it
# against a 216 MiB shortfall. No recompute cost. PYTORCH_ALLOC_CONF is the
# current name (and the one the T4's own error message quoted),
# PYTORCH_CUDA_ALLOC_CONF the older alias.
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Note the two different accounts: GitHub is AmishhYadav, Kaggle is amishyadav123.
REPO_URL   = "https://github.com/AmishhYadav/NeuroVision-X.git"
GIT_REF    = "main"
DATA_SLUG  = "amishyadav123/neurovision-brats-prep"  # attached preprocessed dataset
CKPT_SLUG  = None  # previous notebook output / checkpoint dataset to resume from; None = fresh run

# --- RUN 0: probe v5 -- grad_clip_norm, the last thing before the real runs
#
# v4 SETTLED THE COST QUESTION and its config is reproduced unchanged below,
# because v5 differs only in which commit it clones. Measured at 64^3 with NO
# gradient checkpointing: 1.12 s/step and peak VRAM 6.17 GiB allocated /
# 7.31 GiB reserved of 14.56 -- no OOM, ~7 GiB spare, so checkpointing is off
# for good and its recompute is given back. That is 0.272 h/epoch over 875
# steps, ~21.8 h of training for 80 epochs.
#
# What v5 adds: `train/grad_norm` used to reach W&B only, so reading it meant
# downloading ~1.6 GB of kernel output for a few floats. Trainer now logs a
# per-epoch median / p90 / max / clipped-fraction line to the python log,
# which the Kaggle API returns for free. This run is purely to read that.
#
# THE DECISION IT FEEDS: grad_clip_norm is 1.0 and the measured norm at init
# was 1.63. If clipping fires on most steps, raise it to 5.0 for ALL THREE
# runs before any of them starts -- clipping rescales the whole gradient, so
# two runs clipping at different rates train at different effective
# segmentation learning rates and the P2 ablation stops isolating the
# ambiguity signal. It can never be changed mid-run.
#
# (superseded header kept for context: attempt 4)
# v3 produced the clean number that changed the plan: 3.6 s/step at 96^3 =
# 0.875 h/epoch over 875 steps, so 100 epochs cost ~91 h -- needed twice (the
# model and its P2 ablation) against a 60 h budget. Peak VRAM was 13.59 of
# 14.56 GiB, i.e. 93% of the card, WITH CNN checkpointing already on.
#
# Decision taken: 64^3 patches and 80 epochs, applied to every arm equally via
# configs/experiment/_baseline_common.yaml. The architecture, the fusion, the
# ambiguity gate and the P2 ablation are all untouched -- this is a data and
# schedule change, which is why it lives in the shared file. So patch_size and
# epochs are NOT overridden below; they come from the experiment file now.
#
# What v4 measures, before ~45 h is committed on the strength of it:
#   1. s/step at 64^3 with NO gradient checkpointing. Measured on CPU: 1.46 GB
#      of fp32 activations per patch, so ~5.8 GB for the 4-patch step. The
#      calibrated GPU factor is ~1.0x that plus ~1.6 GB of weights, Adam and
#      workspace (derived from v3: 12.44 GB fp32 predicted vs 13.59 GiB peak
#      observed -- NOT the 0.5-0.6 AMP rule of thumb, which has now been wrong
#      three times). That projects ~7.4 GiB, half the card, so checkpointing
#      should be unnecessary and its ~20-30% recompute can be given back.
#   2. peak VRAM, to confirm 1.
#   3. train/grad_norm, still unsettled -- and it had to be re-measured anyway,
#      since gradient magnitudes change with patch size.
#
# overfit_n=50 keeps this to ~5 minutes: 50 steps/epoch instead of 875, val on
# the same 50 cases. Scale up by h/epoch = s_per_step * 875 / 3600.
# ITS METRICS ARE MEMORIZATION (val == train) AND MUST NEVER BE REPORTED.
EXPERIMENT = "probe_neurovision"

# Offline: a kernel created by `kaggle kernels push` has no WANDB_API_KEY
# attached (Kaggle Secrets are per-notebook and cannot be set from
# kernel-metadata.json). Never "disabled" -- Trainer logs train/grad_norm ONLY
# to W&B, and grad_norm is one of the three things this probe must settle.
WANDB_MODE = "offline"

OVERRIDES  = [
    "+experiment=neurovision",
    # NOTE: no model.encoder.cnn.use_checkpoint here, on purpose. v2 and v3
    # needed it at 96^3; at 64^3 the projection says it is not needed, and
    # this run is the test of that. If it OOMs, add it back -- and then it
    # must be set identically for `neurovision` AND
    # `ablation_content_only_gate`, or their step times differ for a reason
    # that has nothing to do with the ambiguity signal being ablated.
    "data.overfit_n=50",
    "training.epochs=3",
    "training.val_interval=3",  # (epoch + 1) % interval -> validates once, at the end
    "training.scheduler.warmup_epochs=1",  # must be <= epochs or the ramp never completes
    "data.num_workers=2",
    "wandb.group=probes",
    "training.log_interval=5",
]

# --- The three real runs -- replace the block above with one of these ----
# All inherit patch_size 64^3 and epochs 80 from _baseline_common.yaml.
# Each needs WANDB_MODE = "online" plus the WANDB_API_KEY secret attached to
# the notebook in the UI, or "offline" plus a `wandb sync` afterwards.
# NONE of them may set data.overfit_n.
#
# 1. baseline_unet3d (~3 h):
#      EXPERIMENT = "baseline_unet3d"
#      OVERRIDES  = ["+experiment=baseline_unet3d", "data.num_workers=2"]
# 2. neurovision (~23 h, two sessions -- set CKPT_SLUG on the second):
#      EXPERIMENT = None   # the experiment file already names the run
#      OVERRIDES  = ["+experiment=neurovision", "data.num_workers=2"]
# 3. ablation_content_only_gate (~23 h, the load-bearing P2 run). Its
#    checkpointing flags MUST match run 2's exactly:
#      EXPERIMENT = None
#      OVERRIDES  = ["+experiment=ablation_content_only_gate", "data.num_workers=2"]

## 2. Code + dependencies

Clone rather than `pip install git+...` alone: `configs/` and `scripts/` are not package data,
and Hydra needs the config tree on disk. The clone is then installed editable with `--no-deps`,
so `import neurovision` works without `PYTHONPATH` — same as local dev.

`torch`/`torchvision` are stripped from `requirements.txt` on purpose: the Kaggle image ships a
CUDA-matched build, and installing the pinned wheel over it silently loses the GPU. The assert
catches that, and a GPU that was never enabled, in ~30 seconds.

If `pip install -e` ever fails on `requires-python` (Kaggle moving off 3.11), replace that line
with `import sys; sys.path.insert(0, "/kaggle/working/repo/src")`.

In [ ]:
!git clone -q --depth 1 -b {GIT_REF} {REPO_URL} /kaggle/working/repo
# Build the Kaggle install list from requirements.txt minus its own
# `# kaggle-exclude:` line -- single source of truth, no second pinned file to
# drift. Everything excluded is either already in the Kaggle image (and ABI-
# linked to the rest of it) or dev-only.
import pathlib
import re
import subprocess
import sys

_req = pathlib.Path("/kaggle/working/repo/requirements.txt").read_text()
_excl = {w for m in re.findall(r"^#\s*kaggle-exclude:\s*(.+)$", _req, re.M) for w in m.split()}
_keep = [
    ln for ln in _req.splitlines()
    if ln.strip() and not ln.lstrip().startswith("#")
    and re.split(r"[=<>~!\[]", ln.strip())[0].strip() not in _excl
]
print("installing:", " ".join(_keep))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_keep], check=True)

In [ ]:
import torch

# sys.path, NOT `pip install -e`. Kaggle runs Python 3.12 and pyproject.toml
# pins requires-python = ">=3.11,<3.12", so pip REFUSES the editable install
# ("Package 'neurovision-x' requires a different Python"). A `!pip` failure
# does not stop a notebook cell, so that error scrolled past and a previous run
# died four cells later on ModuleNotFoundError. sys.path needs no metadata
# check and works on any interpreter.
sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo/scripts")
import neurovision  # noqa: F401  -- verify HERE, not four cells later
import scipy.ndimage  # noqa: F401 -- canary: breaks if our numpy pin overwrote Kaggle's

assert torch.cuda.is_available(), "No CUDA: GPU accelerator off, or pip replaced Kaggle's CUDA torch build."
_name = torch.cuda.get_device_name(0)
_cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
# is_available() is NOT sufficient, and this is not hypothetical: on a Kaggle
# P100 it returns True while every kernel launch fails, because the stock torch
# build no longer targets sm_60 (min sm_70). Only executing something is honest.
try:
    (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
except Exception as exc:
    raise RuntimeError(
        f"{_name} ({_cap}) reports CUDA available but cannot run a kernel: {exc}\n"
        "Set machine_shape=NvidiaTeslaT4 (sm_75); the P100 is sm_60."
    ) from exc

import monai
import numpy as np

print(f"{_name}  {_cap}")
print(f"torch {torch.__version__} | numpy {np.__version__} | monai {monai.__version__} "
      f"| python {sys.version.split()[0]}")

## 3. Environment — W&B

`WANDB_MODE` in cell 1 picks one of three:

- **`"online"`** — needs `WANDB_API_KEY` as a Kaggle Secret, added under Add-ons → Secrets **and
  attached to this notebook**. A secret on your account but not attached to this kernel fails
  with `No user secrets exist for kernel id <id> and label <label>`. The label is exact and
  case-sensitive; set `SECRET_LABEL` to whatever you actually named it.
- **`"offline"`** — no secret needed. The full run is written to `/kaggle/working/wandb/` and
  saved with the notebook output; `wandb sync <dir>` uploads it later with every metric intact.
- **`"disabled"`** — no logging at all.

Never paste the key into a cell: committed notebook versions are stored with their source.

In [ ]:
import os

# Exact, case-sensitive label of the Kaggle Secret. Must match what you named it
# in Add-ons -> Secrets AND be attached to THIS notebook.
SECRET_LABEL = "WANDB_API_KEY"

# Only "online" needs a Kaggle Secret. "offline" writes the complete run to
# /kaggle/working/wandb/ with no API key at all -- it lands in the notebook
# output, and `wandb sync <dir>` from your Mac uploads it afterwards with the
# curves intact. Use it when the secret is unavailable; it is NOT a downgrade
# in what gets recorded, only in when it appears in the dashboard.
if WANDB_MODE == "online":
    from kaggle_secrets import UserSecretsClient

    # Not wrapped in try/except: a missing secret on a real run means losing the
    # run's curves, so it must stop here, not 11 hours in.
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(SECRET_LABEL)
    print(f"W&B online (secret label {SECRET_LABEL!r})")
elif WANDB_MODE == "offline":
    os.environ["WANDB_DIR"] = "/kaggle/working"
    OVERRIDES = [*OVERRIDES, "wandb.mode=offline"]
    print("W&B OFFLINE -- run written to /kaggle/working/wandb/. "
          "Upload later with: wandb sync <that dir>")
else:
    OVERRIDES = [*OVERRIDES, "wandb.mode=disabled"]
    print("W&B DISABLED -- nothing will be logged.")

## 4. Resolve the attached dataset

Layout is what `scripts/package_for_kaggle.py` builds: `preprocessed/<case>/{image,label}.npy`,
`metadata.csv`, `splits.yaml`. A wrong or unattached dataset raises here, listing what *is*
mounted so the fix is obvious.

In [ ]:
import shutil
from pathlib import Path

# Discovered, not assumed. A dataset does NOT reliably mount at
# /kaggle/input/<slug>: an earlier run of this notebook found the whole of
# /kaggle/input to be just ['datasets'], i.e. one level deeper than the
# documented layout. So look for the shape we need -- a directory holding both
# preprocessed/ and splits.yaml -- across the first few levels, rather than
# hardcoding a path Kaggle is free to change.
_roots = [Path("/kaggle/input")]
_hits = sorted(
    {
        p.parent
        for pat in ("splits.yaml", "*/splits.yaml", "*/*/splits.yaml", "*/*/*/splits.yaml")
        for r in _roots
        for p in r.glob(pat)
        if (p.parent / "preprocessed").is_dir()
    }
)
if len(_hits) != 1:
    _tree = sorted(str(p.relative_to("/kaggle/input")) for p in Path("/kaggle/input").glob("*/*"))
    raise FileNotFoundError(
        f"Expected exactly one dataset with preprocessed/ + splits.yaml under /kaggle/input, "
        f"found {[str(h) for h in _hits]}. Attach {DATA_SLUG}. Present: {_tree[:20]}"
    )
DATA = _hits[0]
PREP, SPLITS = DATA / "preprocessed", DATA / "splits.yaml"
n_cases = sum(1 for p in PREP.iterdir() if p.is_dir())
if n_cases == 0:
    raise FileNotFoundError(f"{PREP} exists but holds no case directories.")
print(n_cases, "preprocessed cases at", PREP)

## 5. Resume

`/kaggle/input` is read-only and `save_checkpoint` must write, so the previous session's
`last.pt` is copied into `/kaggle/working/checkpoints` first. `select_resume_checkpoint` then
finds it there on its own — the training call is identical for a fresh run and a resume.

With `CKPT_SLUG` set, anything other than exactly one `last.pt` raises. Missing it would not
error during training, it would just silently restart from epoch 0 — the expensive failure this
cell exists to prevent.

In [ ]:
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if CKPT_SLUG:
    # Searched across depths, not assumed at /kaggle/input/<slug>/ -- Kaggle
    # mounts sources one level deeper than documented (measured: the data
    # dataset landed at /kaggle/input/datasets/<owner>/<slug>/). Same reason
    # the data cell discovers rather than hardcodes.
    hits = sorted(set(Path("/kaggle/input").glob("**/last.pt")))
    if len(hits) != 1:
        mounted = sorted(str(q.relative_to("/kaggle/input")) for q in Path("/kaggle/input").glob("*/*"))
        raise FileNotFoundError(
            f"CKPT_SLUG={CKPT_SLUG!r}: want exactly one last.pt under /kaggle/input, found "
            f"{[str(h) for h in hits]}. Attach the checkpoint source, or set CKPT_SLUG=None "
            f"to start fresh. Mounted: {mounted[:20]}"
        )
    # /kaggle/input is read-only and save_checkpoint must write, so the file is
    # copied into the writable dir. find_resume_checkpoint then picks it up on
    # its own -- the training call is identical for a fresh run and a resume.
    shutil.copy2(hits[0], CKPT_DIR / "last.pt")
    import torch as _t
    _ck = _t.load(CKPT_DIR / "last.pt", weights_only=True, map_location="cpu")
    print(f"resuming from {hits[0]}")
    print(f"  epoch={_ck['epoch']} -> will start at {_ck['epoch']+1}, "
          f"best {_ck['best_metric_name']}={_ck['best_metric']:.4f}, wandb_run_id={_ck.get('wandb_run_id')}")
    del _ck

## 6. Compose config and train

`hydra.compose` with CLI-style overrides — the same strings `python scripts/train.py a=b` would
take. Calling `run_training` in-process (instead of shelling out) keeps the traceback in the
notebook and lets the cells above hand it already-validated paths.

The log's first line says `FRESH:` or `RESUME: ... from epoch N`. Check it. `max_hours: 11.0`
stops the run cleanly before Kaggle's 12-hour kill.

In [ ]:
import hydra

# sys.path for both src/ and scripts/ was set in the install cell, so this
# import cannot be the first place a missing package shows up.
from train import run_training

overrides = [f"data.root_dir={DATA}", f"data.preprocessing.out_dir={PREP}", f"data.splits.path={SPLITS}",
             f"training.checkpoint.dir={CKPT_DIR}"]
# experiment_name is passed ONLY when EXPERIMENT is set. An explicit
# experiment_name= override always beats a config group's own value, whatever
# the ordering -- Hydra applies group additions during composition and value
# overrides afterwards. So passing it unconditionally would silently rename an
# `+experiment=overfit2` run to EXPERIMENT, sending its checkpoints to
# outputs/<EXPERIMENT> and labelling its W&B run as that experiment. Set
# EXPERIMENT = None whenever OVERRIDES contains a `+experiment=` entry.
if EXPERIMENT is not None:
    overrides.append(f"experiment_name={EXPERIMENT}")
overrides += OVERRIDES

with hydra.initialize_config_dir(version_base="1.3", config_dir="/kaggle/working/repo/configs"):
    cfg = hydra.compose("config", overrides=overrides)
print("experiment_name =", cfg.experiment_name, "| epochs =", cfg.training.epochs)
metrics = run_training(cfg)

## 7. Verify the session output

Training already writes into `/kaggle/working/checkpoints`, which is the only path that survives
into the committed version's output — so there is nothing to copy, only to verify. Duplicating a
754 MB SwinUNETR checkpoint elsewhere under `/kaggle/working` would just eat the ~20 GB quota.

Attach this notebook version's output as input to the next session and set `CKPT_SLUG` to it.

In [ ]:
# Peak VRAM, reported rather than estimated. Three separate pre-run estimates
# of this model's memory were wrong (16 GB assumed capacity vs 14.56 actual,
# a 0.55 AMP factor that is really ~0.75+, and a per-patch figure read as a
# per-step one), so the run itself is now the source of truth. max_memory_*
# are process-lifetime peaks and survive the training call in the same kernel.
_total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(
    f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB allocated, "
    f"{torch.cuda.max_memory_reserved() / 2**30:.2f} GiB reserved, "
    f"of {_total:.2f} GiB total"
)

for name in ("last.pt", "best.pt"):
    p = CKPT_DIR / name
    if not p.is_file():
        raise FileNotFoundError(f"{p} missing — nothing to carry into the next session.")
    print(name, f"{p.stat().st_size / 2**20:.0f} MB  epoch={torch.load(p, weights_only=True)['epoch']}")
print(metrics)